In [10]:

import gurobipy as gp
from gurobipy import GRB
import numpy as np


In [11]:
def solve_lower_level(x_fixed, verbose=False):
    """
    Solve the x-parameterized binary follower problem
    for Exercise 9.15.

    Parameters
    ----------
    x_fixed : list or tuple of length 3
        Fixed leader decision (x1, x2, x3).

    verbose : bool
        Whether to display Gurobi solver output.

    Returns

    phi : float
        Optimal follower objective value.

    y_opt : list
        Optimal binary follower solution.
    """

    if len(x_fixed) != 3:
        raise ValueError("x_fixed must contain 3 values.")

    ll = gp.Model("Lower_Level")
    ll.Params.OutputFlag = int(verbose)

    # Follower variables remain BINARY
    y = ll.addVars(3, vtype=GRB.BINARY, name="y")

    ll.setObjective(4*y[0] + 3*y[1] + 3*y[2], GRB.MAXIMIZE)

    ll.addConstr(4*y[0] + 3*y[1] + 2*y[2] <= 4, name="capacity")
    for i in range(3):
        ll.addConstr(y[i] <= 1 - x_fixed[i], name=f"interdiction_{i+1}")

    ll.optimize()
    if ll.Status != GRB.OPTIMAL:
        raise RuntimeError(f"Lower-level solve failed: {ll.Status}")

    phi = ll.ObjVal
    y_opt = [round(y[i].X) for i in range(3)]

    return phi, y_opt


In [12]:

def solve_refinement(x_fixed, phi, verbose=False):
    """
    Solve the x-parameterized refinement problem
    for Exercise 9.15.

    Parameters
    ----------
    x_fixed : list or tuple
        Integer leader solution x^k.

    phi : float
        Optimal follower value phi(x^k).

    Returns
    -------
    obj : float
        Refined leader objective.

    x_opt, y_opt : list
        Bilevel-feasible solution.
    """

    ref = gp.Model("Refinement")
    ref.Params.OutputFlag = int(verbose)

    x = ref.addVars(3, vtype=GRB.BINARY, name="x")
    y = ref.addVars(3, vtype=GRB.BINARY, name="y")

    profit = 4*y[0] + 3*y[1] + 3*y[2]

    # Leader objective
    ref.setObjective(profit, GRB.MINIMIZE)

    # Leader budget
    ref.addConstr(2*x[0] + x[1] + x[2] <= 2,name="budget")

    # Follower capacity
    ref.addConstr(4*y[0] + 3*y[1] + 2*y[2] <= 4,name="capacity")

    # Follower interdiction constraints
    for i in range(3):
        ref.addConstr(x[i] + y[i] <= 1,name=f"interdiction_{i+1}")

    # Fix all linking variables at x^k
    for i in range(3):
        ref.addConstr(x[i] == round(x_fixed[i]),name=f"fix_x_{i+1}")

    # Enforce follower optimality
    ref.addConstr(profit >= phi, name="follower_optimality")

    ref.optimize()

    if ref.Status == GRB.INFEASIBLE:
        return None

    if ref.Status != GRB.OPTIMAL:
        raise RuntimeError(
            f"Refinement solve failed: {ref.Status}"
        )

    return (
        ref.ObjVal,
        [round(x[i].X) for i in range(3)],
        [round(y[i].X) for i in range(3)]
    )


def get_binding_constraints(model, tol=1e-7):
    if model.Status != GRB.OPTIMAL:
        raise ValueError("Solve the node LP first.")

    variables = model.getVars()
    index = {v.VarName: j for j, v in enumerate(variables)}
    z = np.array([v.X for v in variables])

    binding = []

    # Explicit linear constraints
    for c in model.getConstrs():
        if abs(c.Slack) <= tol:
            row = model.getRow(c)
            coeffs = np.zeros(len(variables))

            for j in range(row.size()):
                v = row.getVar(j)
                coeffs[index[v.VarName]] += row.getCoeff(j)

            # Standardize all inequalities to >=
            if c.Sense == "<":
                coeffs = -coeffs
                rhs = -c.RHS
            elif c.Sense == ">":
                rhs = c.RHS
            else:
                raise ValueError(
                    "Equality constraints need separate handling."
                )

            binding.append((c.ConstrName, coeffs, rhs))

    # Variable bounds (also potentially binding)
    for j, v in enumerate(variables):
        unit = np.zeros(len(variables))
        unit[j] = 1

        if v.LB > -GRB.INFINITY and abs(v.X - v.LB) <= tol:
            binding.append((
                f"{v.VarName}_LB", unit.copy(), v.LB
            ))

        if v.UB < GRB.INFINITY and abs(v.X - v.UB) <= tol:
            binding.append((
                f"{v.VarName}_UB", -unit.copy(), -v.UB
            ))

    return binding



def generate_prop97_cut(model, cut_number, tol=1e-7):
    """
    Generate Proposition 9.7 cut at an integer,
    bilevel-infeasible basic feasible LP solution.

    Assumes:
      - all original variables are integer-constrained
      - all model inequalities have integer data
      - caller verified bilevel infeasibility
    """

    if model.Status != GRB.OPTIMAL:
        raise ValueError("Node LP must be solved first.")

    variables = model.getVars()
    z = np.array([v.X for v in variables])

    # Check integrality
    if np.max(np.abs(z - np.round(z))) > tol:
        raise ValueError("Fractional candidate: branch instead.")

    binding = get_binding_constraints(model, tol)

    if not binding:
        raise ValueError("No binding constraints found.")

    active_matrix = np.array([a for _, a, _ in binding])

    # Check basic feasibility
    if np.linalg.matrix_rank(active_matrix) < len(variables):
        raise ValueError("Candidate is not a basic feasible solution.")

    # Compute alpha, beta, gamma simultaneously
    coefficients = np.sum(active_matrix, axis=0)
    rhs = sum(b for _, _, b in binding)

    # Proposition 9.7 requires integer coefficients
    if (
        np.max(np.abs(coefficients - np.round(coefficients))) > tol
        or abs(rhs - round(rhs)) > tol
    ):
        raise ValueError("Cut data are not integer-valued.")

    coefficients = np.round(coefficients).astype(int)
    rhs = int(round(rhs)) + 1

    # Verify candidate is separated
    if coefficients @ z >= rhs - tol:
        raise ValueError("Cut does not separate the candidate.")

    expression = gp.quicksum(
        int(coefficients[j]) * variables[j]
        for j in range(len(variables))
    )

    cut = model.addConstr(
        expression >= rhs,
        name=f"Prop97_Cut_{cut_number}"
    )

    model.update()

    return cut, coefficients, rhs


Node 0

In [13]:
root99 = gp.Model("Root_Prop99")

x = root99.addVars(3, lb=0, ub=1,vtype=GRB.CONTINUOUS, name="x")

y = root99.addVars(3, lb=0, ub=1,vtype=GRB.CONTINUOUS, name="y")

root99.setObjective(4*y[0] + 3*y[1] + 3*y[2],GRB.MINIMIZE)

root99.addConstr(2*x[0] + x[1] + x[2] <= 2,name="budget")

root99.addConstr(4*y[0] + 3*y[1] + 2*y[2] <= 4,name="capacity")

for i in range(3):
    root99.addConstr(x[i] + y[i] <= 1,name=f"interdiction_{i+1}")

root99.optimize()

print("Root LP objective:", root99.ObjVal)
print("x =", [x[i].X for i in range(3)])
print("y =", [y[i].X for i in range(3)])


Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 5 rows, 6 columns and 12 nonzeros (Min)
Model fingerprint: 0x745281a5
Model has 3 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [3e+00, 4e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 4e+00]

Presolve removed 5 rows and 6 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  0.000000000e+00
Root LP objective: 0.0
x = [0.0, 0.0, 0.0]
y = [0.0, 0.0, 0.0]


In [14]:

def check_bilevel_node(model, UB=float("inf"), tol=1e-6):
    """
    Evaluate a solved LP node for Algorithm 7.

    Checks:
      1. LP bound
      2. Integrality
      3. Follower optimality
      4. Refinement feasibility
      5. Applicability of Proposition 9.9
    """

    if model.Status == GRB.INFEASIBLE:
        print("Node infeasible — fathom")
        return {"action": "fathom_infeasible"}

    if model.Status != GRB.OPTIMAL:
        raise RuntimeError(f"Unexpected Gurobi status: {model.Status}")

    LB = model.ObjVal

    x_val = [
        model.getVarByName(f"x[{i}]").X
        for i in range(3)
    ]
    y_val = [
        model.getVarByName(f"y[{i}]").X
        for i in range(3)
    ]

    print("LP objective:", LB)
    print("x =", x_val)
    print("y =", y_val)

    # Step 1: Bound check
    if LB >= UB - tol:
        print("Fathom by bounding")
        return {"action": "fathom_bound", "LB": LB}

    # Step 2: Integrality check
    fractional = [
        ("x", i, x_val[i])
        for i in range(3)
        if abs(x_val[i] - round(x_val[i])) > tol
    ] + [
        ("y", i, y_val[i])
        for i in range(3)
        if abs(y_val[i] - round(y_val[i])) > tol
    ]

    if fractional:
        print("Fractional variables:", fractional)
        print("Action: branch")
        return {
            "action": "branch_fractional",
            "LB": LB,
            "fractional": fractional
        }

    # Step 3: Follower optimality
    x_binary = [int(round(v)) for v in x_val]

    phi, y_follower = solve_lower_level(x_binary)

    candidate_value = (
        4*y_val[0] + 3*y_val[1] + 3*y_val[2]
    )

    bilevel_feasible = candidate_value >= phi - tol

    print("Follower optimal value:", phi)
    print("Follower solution:", y_follower)
    print("Candidate follower value:", candidate_value)
    print("Bilevel feasible:", bilevel_feasible)

    if bilevel_feasible:
        print("Fathom: bilevel-feasible solution")
        return {
            "action": "fathom_bilevel_feasible",
            "LB": LB,
            "UB_candidate": candidate_value,
            "x": x_binary,
            "y": [int(round(v)) for v in y_val]
        }

    # Step 4: Refinement
    refinement = solve_refinement(x_binary, phi)

    print("Refinement result:", refinement)

    # Assumes solve_refinement returns None iff infeasible.
    prop99_applicable = refinement is None

    print("Proposition 9.9 applicable:", prop99_applicable)

    return {
        "action": (
            "add_prop99_cut"
            if prop99_applicable
            else "branch_linking"
        ),
        "LB": LB,
        "phi": phi,
        "x": x_binary,
        "y": y_val,
        "refinement": refinement,
        "prop99_applicable": prop99_applicable
    }


In [15]:
check_bilevel_node(root99, UB=float("inf"), tol=1e-6)

LP objective: 0.0
x = [0.0, 0.0, 0.0]
y = [0.0, 0.0, 0.0]
Follower optimal value: 4.0
Follower solution: [1, 0, 0]
Candidate follower value: 0.0
Bilevel feasible: False
Refinement result: (4.0, [0, 0, 0], [1, 0, 0])
Proposition 9.9 applicable: False


{'action': 'branch_linking',
 'LB': 0.0,
 'phi': 4.0,
 'x': [0, 0, 0],
 'y': [0.0, 0.0, 0.0],
 'refinement': (4.0, [0, 0, 0], [1, 0, 0]),
 'prop99_applicable': False}

In [7]:

# Node 1
node1_99 = root99.copy()
node1_99.addConstr(
    node1_99.getVarByName("x[0]") <= 0,
    name="branch_x1_0"
)

# Node 2
node2_99 = root99.copy()
node2_99.addConstr(
    node2_99.getVarByName("x[0]") >= 1,
    name="branch_x1_1"
)

# DFS: solve Node 1 first
node1_99.optimize()

if node1_99.Status == GRB.OPTIMAL:
    print("Node 1 objective:", node1_99.ObjVal)
    print("x =", [
        node1_99.getVarByName(f"x[{i}]").X
        for i in range(3)
    ])
    print("y =", [
        node1_99.getVarByName(f"y[{i}]").X
        for i in range(3)
    ])


Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 6 rows, 6 columns and 13 nonzeros (Min)
Model fingerprint: 0x830946dc
Model has 3 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [3e+00, 4e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 4e+00]

Presolve removed 6 rows and 6 columns
Presolve time: 0.01s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  0.000000000e+00
Node 1 objective: 0.0
x = [0.0, 0.0, 0.0]
y = [0.0, 0.0, 0.0]


In [16]:
result1 = check_bilevel_node(node1_99, UB=4)
print(result1["action"])

LP objective: 0.0
x = [0.0, 0.0, 0.0]
y = [0.0, 0.0, 0.0]
Follower optimal value: 4.0
Follower solution: [1, 0, 0]
Candidate follower value: 0.0
Bilevel feasible: False
Refinement result: (4.0, [0, 0, 0], [1, 0, 0])
Proposition 9.9 applicable: False
branch_linking


In [17]:

# Node 3: x2 = 0
node3_99 = node1_99.copy()

node3_99.addConstr(
    node3_99.getVarByName("x[1]") <= 0,
    name="branch_x2_0"
)

# Node 4: x2 = 1
node4_99 = node1_99.copy()

node4_99.addConstr(
    node4_99.getVarByName("x[1]") >= 1,
    name="branch_x2_1"
)

# DFS: Solve Node 3 first
node3_99.optimize()

result3 = check_bilevel_node(node3_99, UB=4)

print("Node 3 action:", result3["action"])


Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 7 rows, 6 columns and 14 nonzeros (Min)
Model fingerprint: 0xfca5f1e8
Model has 3 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [3e+00, 4e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 4e+00]

Presolve removed 7 rows and 6 columns
Presolve time: 0.01s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  0.000000000e+00
LP objective: 0.0
x = [0.0, 0.0, 0.0]
y = [0.0, 0.0, 0.0]
Follower optimal value: 4.0
Follower solution: [1, 0, 0]
Candidate follower value:

In [20]:

# Node 5: x3 = 0
node5_99 = node3_99.copy()

node5_99.addConstr(
    node5_99.getVarByName("x[2]") <= 0,
    name="branch_x3_0"
)

# Node 6: x3 = 1
node6_99 = node3_99.copy()

node6_99.addConstr(
    node6_99.getVarByName("x[2]") >= 1,
    name="branch_x3_1"
)

# DFS: Solve Node 5 first
node5_99.optimize()

result5 = check_bilevel_node(node5_99, UB=4)



Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 8 rows, 6 columns and 15 nonzeros (Min)
Model fingerprint: 0x83a09a85
Model has 3 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [3e+00, 4e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 4e+00]

Presolve removed 8 rows and 6 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.00 seconds (0.00 work units)
Optimal objective  0.000000000e+00
LP objective: 0.0
x = [0.0, 0.0, 0.0]
y = [0.0, 0.0, 0.0]
Follower optimal value: 4.0
Follower solution: [1, 0, 0]
Candidate follower value:

All linking vars are fixed at noe 5. So Fathomed

In [ ]:

node6_99.optimize()

result6 = check_bilevel_node(node6_99, UB=4)


Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 8 rows, 6 columns and 15 nonzeros (Min)
Model fingerprint: 0xa27f9156
Model has 3 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [3e+00, 4e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 4e+00]

Presolve removed 8 rows and 6 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  0.000000000e+00
LP objective: 0.0
x = [0.0, 0.0, 1.0]
y = [0.0, 0.0, 0.0]
Follower optimal value: 4.0
Follower solution: [1, 0, 0]
Candidate follower value:

In [22]:

node4_99.optimize()

result4 = check_bilevel_node(node4_99, UB=4)

print("Node 4 action:", result4["action"])


Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 7 rows, 6 columns and 14 nonzeros (Min)
Model fingerprint: 0x092aed4a
Model has 3 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [3e+00, 4e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 4e+00]

Presolve removed 7 rows and 6 columns
Presolve time: 0.01s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  0.000000000e+00
LP objective: 0.0
x = [0.0, 1.0, 0.0]
y = [0.0, 0.0, 0.0]
Follower optimal value: 4.0
Follower solution: [1, 0, 0]
Candidate follower value:

In [23]:

# Node 7: x3 = 0
node7_99 = node4_99.copy()

node7_99.addConstr(
    node7_99.getVarByName("x[2]") <= 0,
    name="branch_x3_0"
)

# Node 8: x3 = 1
node8_99 = node4_99.copy()

node8_99.addConstr(
    node8_99.getVarByName("x[2]") >= 1,
    name="branch_x3_1"
)

# DFS: Solve Node 7 first
node7_99.optimize()

result7 = check_bilevel_node(node7_99, UB=4)

print("Node 7 action:", result7["action"])


Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 8 rows, 6 columns and 15 nonzeros (Min)
Model fingerprint: 0xaf7d436a
Model has 3 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [3e+00, 4e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 4e+00]

Presolve removed 8 rows and 6 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  0.000000000e+00
LP objective: 0.0
x = [0.0, 1.0, 0.0]
y = [0.0, 0.0, 0.0]
Follower optimal value: 4.0
Follower solution: [1, 0, 0]
Candidate follower value:

In [24]:

node8_99.optimize()

result8 = check_bilevel_node(node8_99, UB=4)

print("Node 8 action:", result8["action"])


Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 8 rows, 6 columns and 15 nonzeros (Min)
Model fingerprint: 0x0a89baa9
Model has 3 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [3e+00, 4e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 4e+00]

Presolve removed 8 rows and 6 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  0.000000000e+00
LP objective: 0.0
x = [0.0, 1.0, 1.0]
y = [0.0, 0.0, 0.0]
Follower optimal value: 4.0
Follower solution: [1, 0, 0]
Candidate follower value:

In [26]:

# Node 2 was previously created from root99
node2_99.optimize()

result2 = check_bilevel_node(node2_99, UB=4)



Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 6 rows, 6 columns and 13 nonzeros (Min)
Model has 3 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [3e+00, 4e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 4e+00]


Solved in 0 iterations and 0.00 seconds (0.00 work units)
Optimal objective  0.000000000e+00
LP objective: 0.0
x = [1.0, 0.0, 0.0]
y = [0.0, 0.0, 0.0]
Follower optimal value: 3.0
Follower solution: [0, 1, 0]
Candidate follower value: 0.0
Bilevel feasible: False
Refinement result: (3.0, [1, 0, 0], [0, 0, 1])
Proposition 9.9 applicable: False


x1=1 implies all the linking variables are fixed since x2 and x3 needs to be zero because of the constraint in the problem. Therefore this node can also be pruned
